In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model="gpt-5-nano", tools=[], checkpointer=InMemorySaver())

result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕 내이름은 김한호야."}]},
    {"configurable": {"thread_id": "hanho"}},
)

In [3]:
result1["messages"][-1].content

'안녕하세요, 김한호님. 만나서 반가워요! 오늘 어떤 도움을 원하시는지 말씀해 주세요. 예를 들어 한국어 공부, 글쓰기나 번역, 정보 검색, 코딩 도움 등 무엇이든 도와드리겠습니다. 원하시면 저를 편하게 부를 이름도 말씀해 주세요. 어떤 주제로 시작해 볼까요?'

In [4]:
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "내이름이 뭐라고했지?"}]},
    {"configurable": {"thread_id": "hanho"}},
)

In [5]:
result2["messages"][-1].content

'김한호라고 말씀하셨어요.  \n다른 이름으로 불러드릴까요?'

In [6]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace = ("user", "minsu", "preferences")

store.put(namespace, "language", {"value": "ko"})

result = store.get(namespace, "language")
result

Item(namespace=['user', 'minsu', 'preferences'], key='language', value={'value': 'ko'}, created_at='2026-04-20T04:37:54.370991+00:00', updated_at='2026-04-20T04:37:54.370993+00:00')

In [7]:
# 장기메모리 실전예제 - 추천시스템

from dataclasses import dataclass
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

In [8]:
@dataclass
class UserContext:
    user_id: str

In [9]:
store = InMemoryStore()

In [10]:
@tool
def save_preference(
    category: str, value: str, runtime: ToolRuntime[UserContext]
) -> str:
    """
    사용자의 선호 정보를 저장합니다.
    예)  category='drink', valuse='tea'
    """
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    runtime.store.put(namespace, category, {"value": value})

In [11]:
@tool
def load_preference(category: str, runtime: ToolRuntime[UserContext]) -> str:
    """사용자의 선호 정보를 불러옵니다."""

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    memory = runtime.store.get(namespace, category)

    if memory is None:
        return f"{category}에 대한 저장된 정보가 없습니다."

    return f"{category}에 대한 정보는 {memory.value['value']} 입니다."

In [12]:
@tool
def recommend_by_preference(runtime: ToolRuntime[UserContext]) -> str:
    """저장된 선호를 바탕으로 간단한 추천을 합니다."""

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    drink_memory = runtime.store.get(namespace, "drink")
    food_memory = runtime.store.get(namespace, "food")

    drink = drink_memory.value["value"] if drink_memory else None
    food = food_memory.value["value"] if food_memory else None

    if drink and food:
        return (
            f"당신은 {drink}와 {food}를 좋아하므로, {food}와 함께 {drink}를 추천합니다."
        )
    elif drink:
        return f"당신은 {drink}를 좋아하므로, 오늘은 {drink}를 추천합니다."
    elif food:
        return f"당신은 {food}를 좋아하므로, {food} 관련 메뉴를 추천합니다."
    else:
        return "저장된 선호 정보가 없어서 아직 추천하기 어렵습니다."

In [13]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[recommend_by_preference, load_preference, save_preference],
    store=store,
)

In [14]:
user_context = UserContext(user_id="hanho")

response1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "나는 drink는 tea를 좋아하고 food는 pasta를 좋아해",
            }
        ]
    },
    context=user_context,
)

response1

{'messages': [HumanMessage(content='나는 drink는 tea를 좋아하고 food는 pasta를 좋아해', additional_kwargs={}, response_metadata={}, id='e8160c04-553c-4f1e-9cc3-654ba66f6e56'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 768, 'prompt_tokens': 206, 'total_tokens': 974, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWagMc9aIr9KhWN1bjdaQlT3QBcir', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da92e-429c-70e0-b528-eb5c4226d112-0', tool_calls=[{'name': 'save_preference', 'args': {'category': 'drink', 'value': 'tea'}, 'id': 'call_ZlM7Ye8lPW0DPuXbghdtJ46t', 'type': 'tool_call'}, {'name': 'save_preference', 'args': {'categ

In [15]:
# 2) 저장된 정보 조회
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "내가 어떤 drink를 좋아한다고 했지?"}]},
    context=user_context,
)

In [16]:
response2["messages"][-1].content

'맞아요. drink 선호로 tea를 기억하고 있어요.\n\n원하시면 tea에 맞춘 간단한 추천을 드릴게요. 또 차의 종류(녹차, 홍차, 우롱차, 허브차 등)나 강도, 설탕/우유 여부 같은 세부 선호를 저장해둘 수도 있어요. 어떤 방식으로 도와드릴까요? 원하시면 지금 바로 추천을 드리겠습니다.'

In [17]:
namespace = ("users", "hanho", "preference")
result = store.get(namespace, "drink")
result

Item(namespace=['users', 'hanho', 'preference'], key='drink', value={'value': 'tea'}, created_at='2026-04-20T04:38:04.045613+00:00', updated_at='2026-04-20T04:38:04.045615+00:00')

In [18]:
from duckduckgo_search import DDGS


def search_movies(query: str, max_results: int = 5):
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            results.append(r["title"])
    return results

In [26]:
@tool
def recommend_movie_with_search(runtime: ToolRuntime[UserContext]) -> str:
    """DuckDuckGo 검색 기반 영화 추천"""

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preference")

    genre_memory = runtime.store.get(namespace, "genre")
    mood_memory = runtime.store.get(namespace, "mood")

    genre = genre_memory.value["value"] if genre_memory else None
    mood = mood_memory.value["value"] if mood_memory else None

    # 1. 검색 쿼리 생성
    query_parts = []

    if genre:
        query_parts.append(f"{genre} 영화")
    if mood:
        query_parts.append(f"{mood} 영화")

    if not query_parts:
        return "취향 정보가 없어서 추천하기 어렵습니다."

    query = " ".join(query_parts) + " 추천"

    # 2. DuckDuckGo 검색
    results = search_movies(query)

    # 3. 결과 반환
    if results:
        return f"검색 기반 추천 ({query}):\n" + "\n".join([f"- {r}" for r in results])
    else:
        return "검색 결과에서 영화를 찾지 못했습니다."

In [32]:
@tool
def save_movie_preference(
    runtime: ToolRuntime[UserContext], genre: str = None, mood: str = None
):
    """사용자의 영화 취향을 저장합니다."""

    namespace = ("users", runtime.context.user_id, "preference")

    if genre:
        runtime.store.put(namespace, "genre", {"value": genre})
    if mood:
        runtime.store.put(namespace, "mood", {"value": mood})
    print("취향이 저장되었습니다. 프린트")
    return "취향이 저장되었습니다."

In [34]:
@tool
def load_movie_preference(runtime: ToolRuntime[UserContext]):
    """사용자의 영화 장르 선호를 불러옵니다"""
    namespace = ("users", runtime.context.user_id, "preference")

    genre = runtime.store.get(namespace, "genre")
    mood = runtime.store.get(namespace, "mood")

    return {
        "genre": genre.value["value"] if genre else None,
        "mood": mood.value["value"] if mood else None,
    }

In [35]:
agent_d = create_agent(
    model="gpt-5-nano",
    tools=[recommend_movie_with_search, save_movie_preference, load_movie_preference],
    store=store,
)

In [37]:
user_context_kb = UserContext(user_id="kb")

response1_1 = agent_d.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "나는 인터스텔라같은 우주 영화 장르를 좋아해 기억해줘",
            }
        ]
    },
    context=user_context_kb,
)

response1_1

취향이 저장되었습니다. 프린트


{'messages': [HumanMessage(content='나는 인터스텔라같은 우주 영화 장르를 좋아해 기억해줘', additional_kwargs={}, response_metadata={}, id='f4f8f276-13bc-4fad-b15f-88eaaedcd1df'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 546, 'prompt_tokens': 204, 'total_tokens': 750, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWar43HoDtIOKwauq0EnmFWusOzj8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da938-63e8-70c3-8b90-eed4c11fcc34-0', tool_calls=[{'name': 'save_movie_preference', 'args': {'genre': 'space / sci-fi', 'mood': 'thought-provoking'}, 'id': 'call_SZfx7Ug2B7ullioe9l0S4qvh', 'type': 'tool_call'}], invalid_tool_calls=[], u

In [38]:
response1_2 = agent_d.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "내가 어떤 영화를 좋아한다고 했지?",
            }
        ]
    },
    context=user_context_kb,
)

response1_2

C:\Users\user\AppData\Local\Temp\ipykernel_10496\2008987970.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


{'messages': [HumanMessage(content='내가 어떤 영화를 좋아한다고 했지?', additional_kwargs={}, response_metadata={}, id='8f3fd6c1-8f58-4d26-9eb4-0e917ec6f3f5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 196, 'total_tokens': 409, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWaxR1D8Xufdmldg9nq0kqP9jNrkw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da93e-69bb-7051-bcb1-91a04f59945a-0', tool_calls=[{'name': 'load_movie_preference', 'args': {}, 'id': 'call_uha6Ukj0YLvlQv08wjjGf1wQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens': 213, 'total

In [40]:
namespace = ("users", "kb", "preference")
store.get(namespace, "genre")

Item(namespace=['users', 'kb', 'preference'], key='genre', value={'value': 'space / sci-fi'}, created_at='2026-04-20T04:49:04.764316+00:00', updated_at='2026-04-20T04:49:04.764318+00:00')